# GP08 — Buffer Analysis: Ενδεικτική λύση

Τα buffers δημιουργούνται σε EPSG:2100, ώστε οι αποστάσεις να είναι μέτρα. Η λύση κρατά individual service buffers, road buffer και spatially selected services χωριστά, ώστε να διακρίνεται η νέα geometry από το αρχικό dataset.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = Path('data/gp08')
services_df = pd.read_csv(root / 'services.csv')
services = gpd.GeoDataFrame(services_df, geometry=gpd.points_from_xy(services_df.longitude, services_df.latitude), crs='EPSG:4326')
roads = gpd.read_file(root / 'roads.geojson')
area = gpd.read_file(root / 'study_area.geojson')
target_crs = 'EPSG:2100'
services_m, roads_m, area_m = (x.to_crs(target_crs) for x in (services, roads, area))


In [ ]:
school_buffers = services_m[services_m['service_type'].eq('school')].copy()
school_buffers['geometry'] = school_buffers.geometry.buffer(500)
road_buffer = gpd.GeoDataFrame({'road_id': roads_m['road_id'], 'geometry': roads_m.geometry.buffer(250)}, crs=target_crs)
print('school buffers:', len(school_buffers), school_buffers.geom_type.unique())
print('road buffers:', len(road_buffer), road_buffer.geom_type.unique())


In [ ]:
near_school = gpd.sjoin(services_m, school_buffers[['service_id','geometry']], predicate='within', how='inner', lsuffix='service', rsuffix='buffer')
print(near_school[['service_id_service','name','service_type']])
print('school buffer bounds:', school_buffers.total_bounds)


In [ ]:
assert services_m.crs.to_epsg() == 2100
assert roads_m.crs.to_epsg() == 2100
assert area_m.crs.to_epsg() == 2100
assert len(school_buffers) == 2
assert school_buffers.geometry.geom_type.eq('Polygon').all()
assert road_buffer.geometry.geom_type.eq('Polygon').all()
assert school_buffers.geometry.notna().all()
assert school_buffers.total_bounds[0] < school_buffers.total_bounds[2]


In [ ]:
school_buffers.to_file('gp08_buffers.gpkg', layer='school_buffers', driver='GPKG')
road_buffer.to_file('gp08_buffers.gpkg', layer='road_buffers', driver='GPKG')
print('exported buffers')
